# Evaluation on Kaggle (phase 3)

Thin wrapper (CLAUDE.md §4): installs the `t2sql` package from GitHub, rebuilds the dataset splits from the templates, then runs `t2sql.eval.run` once per model. No logic lives here.

**Before running**
- Settings → Accelerator: **GPU T4**; Internet: **On** (GitHub and the Hugging Face models).
- Add Input → the private dataset holding `eco2mix.duckdb` (it is not in git).

**How to run**: first run every cell down to the short test, by hand, and check the batch times it prints. Only then *Save Version → Save & Run All*. Raw outputs are appended batch by batch and a new launch resumes where the last one stopped.

**After running**: download `predictions.zip` from the Output tab and unzip it into `results/` of the repository.

In [ ]:
REPO = "https://github.com/zaninip/domain-text2sql"
COMMIT = "main"  # pin a commit SHA to reproduce a run exactly
MODELS = ["Qwen/Qwen3-1.7B", "Qwen/Qwen2.5-Coder-1.5B-Instruct"]
SPLIT = "val"  # val for the model pilot; test for the baseline
MODE = "zero_shot"
SMOKE_LIMIT = 16  # two batches of 8: enough to time a batch and catch a hang

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q {REPO} /kaggle/working/repo
%cd /kaggle/working/repo
!git checkout -q {COMMIT} && git log -1 --oneline
!pip install -q -e ".[model]"

In [ ]:
import torch

# run.py refuses to start without a GPU; fail here, before anything slow
assert torch.cuda.is_available(), "torch sees no GPU: check Settings → Accelerator"
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
import glob
import shutil
from pathlib import Path

found = glob.glob("/kaggle/input/**/eco2mix.duckdb", recursive=True)
assert found, "attach the dataset holding eco2mix.duckdb (Add Input)"
Path("data").mkdir(exist_ok=True)
shutil.copy(found[0], "data/eco2mix.duckdb")

In [ ]:
# Same command as `make dataset`; expected: train 2126 / val 402 / test 490
!python -u -m t2sql.dataset.build

In [ ]:
import os
import subprocess

# Progress bars write without newlines and turn a remote log into one unreadable line
os.environ |= {"TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1"}


def evaluate(model: str, *extra: str) -> None:
    """One run of t2sql.eval.run; check=True stops the notebook if it fails."""
    command = ["python", "-u", "-m", "t2sql.eval.run", "--model", model]
    subprocess.run([*command, "--split", SPLIT, "--mode", MODE, *extra], check=True)

**Short test**: the first model on the first examples, written to `results/smoke` and deleted first, so it always generates on the GPU and never touches `results/predictions`. Read the `batch` lines: a batch of 8 should take seconds, not minutes.

In [ ]:
SMOKE_DIR = "results/smoke"
shutil.rmtree(SMOKE_DIR, ignore_errors=True)
evaluate(MODELS[0], "--limit", str(SMOKE_LIMIT), "--out", SMOKE_DIR)

**Full run**: every model on the whole split (resumes what is already generated in `results/predictions`).

In [ ]:
for model in MODELS:
    evaluate(model)

In [ ]:
import json

keys = ["execution_accuracy", "valid_sql_rate", "mean_output_tokens"]
keys += ["generation_minutes", "outcomes"]
for path in sorted(Path("results/predictions").glob("*.summary.json")):
    summary = json.loads(path.read_text())
    print(summary["model"], summary["split"], summary["mode"], summary.get("device"))
    for key in keys:
        print(f"  {key}: {summary.get(key)}")

In [ ]:
!zip -qr /kaggle/working/predictions.zip results/predictions
!ls -lh /kaggle/working/predictions.zip